In [1]:
import os
print(os.path.exists("../data/raw_close_prices.csv"))   # must print True

True


In [2]:
import pandas as pd
import numpy as np

close = pd.read_csv("../data/raw_close_prices.csv", index_col=0, parse_dates=True)
print("Raw shape:", close.shape)
print(close.isna().sum())

# MSFT's trading days are the master calendar
close = close.dropna(subset=["MSFT"])

# Forward-fill short gaps only (never back-fill: that would use future data)
close = close.ffill(limit=3)
close = close.dropna()

print("Clean shape:", close.shape)
print("Remaining NaNs:", close.isna().sum().sum())   # should be 0

Raw shape: (4384, 8)
MSFT      154
IBM       154
GOOGL     154
USDJPY      6
GBPUSD      6
SP500     154
DOW       154
VIX       152
dtype: int64
Clean shape: (4230, 8)
Remaining NaNs: 0


In [3]:
weekly_close = close.resample("W-FRI").last()

# Drop the final row if it is an unfinished week (today is not Friday)
if close.index[-1] < weekly_close.index[-1]:
    weekly_close = weekly_close.iloc[:-1]

weekly_ret = weekly_close.pct_change().dropna()
print(weekly_ret.shape)
print(weekly_ret.describe().round(4))

(876, 8)
           MSFT       IBM     GOOGL    USDJPY    GBPUSD     SP500       DOW  \
count  876.0000  876.0000  876.0000  876.0000  876.0000  876.0000  876.0000   
mean     0.0041    0.0021    0.0042    0.0007   -0.0001    0.0024    0.0021   
std      0.0334    0.0346    0.0385    0.0127    0.0120    0.0223    0.0220   
min     -0.1437   -0.2604   -0.1292   -0.0449   -0.0856   -0.1498   -0.1730   
25%     -0.0151   -0.0145   -0.0191   -0.0069   -0.0073   -0.0083   -0.0086   
50%      0.0040    0.0015    0.0045    0.0010   -0.0004    0.0034    0.0029   
75%      0.0218    0.0193    0.0244    0.0083    0.0076    0.0145    0.0134   
max      0.2175    0.1732    0.2581    0.0639    0.0614    0.1210    0.1284   

            VIX  
count  876.0000  
mean     0.0128  
std      0.1711  
min     -0.4266  
25%     -0.0852  
50%     -0.0111  
75%      0.0780  
max      1.3484  


In [4]:
# Target: MSFT's return NEXT week
target = weekly_ret["MSFT"].shift(-1).rename("MSFT_next_week_ret")

# Features: this week's and the past 4 weeks' returns for every asset
lags = [0, 1, 2, 3, 4]
features = pd.concat(
    [weekly_ret.shift(lag).add_suffix(f"_lag{lag}") for lag in lags],
    axis=1
)

dataset = pd.concat([features, target], axis=1).dropna()
print(dataset.shape)
print(dataset.columns.tolist())

(871, 41)
['MSFT_lag0', 'IBM_lag0', 'GOOGL_lag0', 'USDJPY_lag0', 'GBPUSD_lag0', 'SP500_lag0', 'DOW_lag0', 'VIX_lag0', 'MSFT_lag1', 'IBM_lag1', 'GOOGL_lag1', 'USDJPY_lag1', 'GBPUSD_lag1', 'SP500_lag1', 'DOW_lag1', 'VIX_lag1', 'MSFT_lag2', 'IBM_lag2', 'GOOGL_lag2', 'USDJPY_lag2', 'GBPUSD_lag2', 'SP500_lag2', 'DOW_lag2', 'VIX_lag2', 'MSFT_lag3', 'IBM_lag3', 'GOOGL_lag3', 'USDJPY_lag3', 'GBPUSD_lag3', 'SP500_lag3', 'DOW_lag3', 'VIX_lag3', 'MSFT_lag4', 'IBM_lag4', 'GOOGL_lag4', 'USDJPY_lag4', 'GBPUSD_lag4', 'SP500_lag4', 'DOW_lag4', 'VIX_lag4', 'MSFT_next_week_ret']


In [6]:
msft_w = weekly_close["MSFT"]

tech = pd.DataFrame(index=weekly_close.index)
tech["MSFT_MA4_ratio"]   = msft_w / msft_w.rolling(4).mean() - 1
tech["MSFT_MA12_ratio"]  = msft_w / msft_w.rolling(12).mean() - 1
tech["MSFT_EMA12_ratio"] = msft_w / msft_w.ewm(span=12, adjust=False).mean() - 1
tech["MSFT_momentum4"]   = msft_w.pct_change(4)
tech["MSFT_vol4"]        = weekly_ret["MSFT"].rolling(4).std()

dataset_tech = pd.concat([features, tech, target], axis=1, sort=False).dropna()
print(dataset_tech.shape)

(865, 46)
